<div align="center">
  <h1>📖 Fable Demo</h1>
</div>

<div style="
    background: #1E2A38;
    color: #EEEEEE;
    padding: 16px;
    border-radius: 8px;
    margin-bottom: 20px;
    border-left: 3px solid #FECB52;
    text-align: center;
">
    Compact JAX + Flax language model for playful short stories.
</div>

---


## 1. Installation

Install Fable straight from GitHub. 

If doing model training, uncomment the second `pip install` and run with a GPU runtime enabled.


In [ ]:
# Install the latest Fable build
!pip install --quiet git+https://github.com/auxeno/fable

# Optional: install a GPU-enabled JAX wheel (not needed for text generation)
# !pip install --quiet "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

### Verify the runtime

JAX automatically picks TPU/GPU devices when they're available.


In [ ]:
import fable, jax
print('Fable version:', fable.__version__)
jax.devices()

## 2. Generate stories

Use the Python text generation API:


In [ ]:
from fable.generate import generate_text

generate_text("Lily got a new puppy", temperature=0.6)

### Temperature sweep

Lower temperatures stick closely to the training distribution. Higher values explore stranger continuations.


In [ ]:
for temp in (0.4, 0.6, 0.8):
    print(f"\n--- temperature {temp} ---")
    generate_text("Lily got a new puppy", temperature=temp, seed=42)

## 3. Train

### Prepare TinyStories data

Run the full download → clean → tokenise pipeline. This writes files into the `data/` directory.

> ⚠️ Downloads ~250 MB and takes a few minutes the first time.


In [ ]:
from fable.data import prepare_tinystories_dataset

prepare_tinystories_dataset()

### Quick training run

Train a smaller configuration for a single epoch to verify the pipeline. Increase `num_epochs` for longer runs.

> 💡 Ensure the TinyStories pipeline above has completed before running this cell.


In [ ]:
import jax
from flax import nnx
from fable.config import GPTConfig
from fable.model import GPT
from fable.train import train

small_config = GPTConfig(
    num_layers=2,
    embed_dim=128,
    num_heads=4,
    max_seq_len=128,
    batch_size=32,
    num_epochs=1,
    enable_checkpointing=False,
    verbose=True,
)

rng = jax.random.PRNGKey(small_config.seed)
scratch_model = GPT(config=small_config, rngs=nnx.Rngs(rng))
trained_model = train(model=scratch_model)

### Sample from the trained model

Use the freshly trained weights to generate a story.


In [ ]:
from fable.generate import generate_text

generate_text("Lily got a new puppy", model=trained_model, temperature=0.6)